# Managing State in AutoGen

By default an agent or team forgets everything when your script ends. **State management** lets you save the conversation and reload it later.

Two methods, both on agents *and* teams:
- `await x.save_state()` → returns a `dict` you can persist (e.g. to JSON).
- `await x.load_state(state)` → restores from that dict.

Use `await x.reset()` to wipe state and start fresh.

## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

agent = AssistantAgent(
    name="chatbot",
    model_client=model_client,
    system_message="You are a helpful assistant. Keep replies short.",
)

## 1. Have a short conversation

In [3]:
r = await agent.on_messages(
    [TextMessage(content="My name is Pradeep.", source="user")],
    cancellation_token=CancellationToken(),
)
print(r.chat_message.content)

Nice to meet you, Pradeep! How can I assist you today?


## 2. Save the agent's state

`save_state()` returns a plain dict — it captures the conversation history so the agent can pick up later.

In [4]:
state = await agent.save_state()
print(type(state).__name__)
print(list(state.keys()))

dict
['type', 'version', 'llm_context']


## 3. Persist to a JSON file

The state dict is JSON-serializable, so you can write it to disk and reload it next session.

In [5]:
import json

with open("agent_state.json", "w") as f:
    json.dump(state, f, indent=2)

with open("agent_state.json") as f:
    saved = json.load(f)

print("Saved.")

Saved.


## 4. Load state into a fresh agent

Build a brand-new agent and pour the saved state into it. It will remember the earlier conversation.

In [6]:
fresh_agent = AssistantAgent(
    name="chatbot",
    model_client=model_client,
    system_message="You are a helpful assistant. Keep replies short.",
)

await fresh_agent.load_state(saved)

r = await fresh_agent.on_messages(
    [TextMessage(content="What is my name?", source="user")],
    cancellation_token=CancellationToken(),
)
print(r.chat_message.content)

Your name is Pradeep.


## 5. Reset to clear state

When you want a clean slate without rebuilding the agent, call `reset()`.

In [7]:
await fresh_agent.on_reset(cancellation_token=CancellationToken())

r = await fresh_agent.on_messages(
    [TextMessage(content="What is my name?", source="user")],
    cancellation_token=CancellationToken(),
)
print(r.chat_message.content)

I don't know your name. Can you tell me?


## Teams work the same way

`team.save_state()` / `team.load_state(state)` / `await team.reset()` — same pattern, but the dict also includes the state of every participant agent.

In [8]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

writer = AssistantAgent(name="writer", model_client=model_client,
                       system_message="Write one sentence on the topic.")
critic = AssistantAgent(name="critic", model_client=model_client,
                       system_message="Reply APPROVE or one suggestion.")

team = RoundRobinGroupChat([writer, critic], termination_condition=MaxMessageTermination(4))

await team.run(task="Explain what an AI agent is in one line.")
team_state = await team.save_state()
print("Team state keys:", list(team_state.keys()))

Team state keys: ['type', 'version', 'agent_states']


In [ ]:
import json

with open("agent_state_agents.json", "w") as f:
    json.dump(team_state, f, indent=2)

with open("agent_state_agents.json") as f:
    saved = json.load(f)

print("Saved.")

In [ ]:
await model_client.close()